# 1. Import Libraries and Download Datasets

In [1]:
!pip install contractions emoji emot

from pprint import pprint
import pathlib
import csv
import os
import csv
import contractions
import shutil
import emoji
from emot.emo_unicode import EMOTICONS_EMO
import re
import string

import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')

preprocessed_data_path = "../data/preprocessed"

STOPWORDS = [
"a","about","above","after","again","against","ain" "ain't","all","am","an","and","any","are",
"aren","aren't","as","at","be","because","been","before","being","below","between",
"both","but","by","can","can't","cannot","could","couldn","couldn't","d","did","didn",
"didn't","do","does","doesn","doesn't","doing","don","don't","down","during","each",
"few","for","from","further","had","hadn","hadn't","has","hasn","hasn't","have",
"haven","haven't","having","he","he'd","he'll","he's","her","here","hers","herself",
"him","himself","his","how","how's","i","i'd","i'll","i'm","i've","if","in","into",
"is","isn","isn't","it","it's","its","itself","just","ll","m","ma","me","mightn",
"mightn't","more","most","mustn","mustn't","my","myself","needn","needn't","no",
"nor","not","now","o","of","off","on","once","only","or","other","our","ours",
"ourselves","out","over","own","re","s","same","shan","shan't","she","she'd",
"she'll","she's","should","should've","shouldn","shouldn't","so","some","such",
"t","than","that","that'll","the","their","theirs","them","themselves","then",
"there","these","they","they'd","they'll","they're","they've","this","those",
"through","to","too","under","until","up","'ve","'am","'s","ve","very","was","wasn","wasn't","we",
"we'd","we'll","we're","we've","were","weren","weren't","what","what's","when",
"when's","where","where's","which","while","who","who's","whom","why","why's",
"will","with","won","won't","would","wouldn","wouldn't","y","you","you'd","you'll",
"you're","you've","your","yours","yourself","yourselves"
]



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


[nltk_data] Downloading package stopwords to /home/lenovo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/lenovo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /home/lenovo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/lenovo/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# 2. Get Paths and Create Functions

In [34]:
# create preprocessed dir if not exists
pathlib.Path(preprocessed_data_path).mkdir(parents=True, exist_ok=True)

# copy raw reviews for preprocessing
shutil.copytree(raw_data_path, preprocessed_data_path, dirs_exist_ok=True)

def preprocess_data(df, process_fn):
    df["content"] = df["content"].apply(process_fn)
    
    return df

# 3. Expand Contractions

In [35]:
contractions_dict = {
    "i'mma": "i will"
}

def expand_contractions(content):
    expanded_words = []
    content_words = content.split(" ")
    for word in content_words:
        word_new = ""
        if word not in contractions_dict.keys():
            word_new = contractions.fix(word)
        else:
            word_new = contractions_dict[word]
        expanded_words.append(word_new)
    return " ".join(expanded_words)

# 4. Remove Emojis

In [36]:
def remove_emojis(content):
    return emoji.replace_emoji(content, replace="")

# 5. Remove Emoticons

In [37]:
emoticons_dict_custom = EMOTICONS_EMO
emoticons_dict_custom["¯\\_(ツ)_/¯"] = "Shrug"

emoticon_regex = re.compile(
    "|".join(map(re.escape, emoticons_dict_custom.keys()))
)

def remove_emoticons(content):
    return re.sub(emoticon_regex, "", content)

# 6. Remove Stopwords

In [38]:
def remove_stopwords(content):
    content_words = content.split(" ")
    filtered = [w for w in content_words if w not in STOPWORDS]

    filtered_initial_text = " ".join(filtered)

    # do it again with nltk
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(filtered_initial_text.lower())

    filtered_tokens = [word for word in tokens if word not in stop_words]

    return " ".join(filtered_tokens)

# 7. Lemmatise

In [39]:
lemmatizer = WordNetLemmatizer()

def lemmatize(content):
    words = content.split(" ")
    lemmatized_words = [lemmatizer.lemmatize(word, 'v') for word in words]
    text = " ".join(lemmatized_words)
    return text

# 8. Remove Punctuations

In [40]:
def remove_punctuation(content):
    # replace punctuation with space
    translator = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
    text = content.translate(translator)
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# 9. Run All Functions

In [41]:
for name in os.listdir(preprocessed_data_path):
    path = os.path.join(preprocessed_data_path, name)

    df = pd.read_csv(path)

    df = preprocess_data(df, expand_contractions)
    df = preprocess_data(df, remove_emojis)
    df = preprocess_data(df, remove_emoticons)
    df = preprocess_data(df, remove_stopwords)
    df = preprocess_data(df, lemmatize)
    df = preprocess_data(df, remove_punctuation)

    df.to_csv(path, index=False)

# 10. Print Results

In [42]:
for name in os.listdir(preprocessed_data_path):
    path = os.path.join(preprocessed_data_path, name)

    with open(path, newline="") as f:
        df = pd.read_csv(path)
        print(df["content"])

0       steam app work fine years log recently whwn go...
1       great stop work every awhile log relog due ser...
2       ability access reset account go great especial...
3       pretty damn hard use website mobile easier con...
4       try well hour log app change password account ...
                              ...                        
9995    possible stream game tablet phone desktop clie...
9996    app rubbish firstly game phone gmail account s...
9997    please fix sign error communicate steam server...
9998    app call incorrect password matter many time r...
9999    please ad make account button never make downl...
Name: content, Length: 10000, dtype: str
